# Diabetes model · Test interface

A small form to try the saved model on a single patient.

**How to use**
1. Run the three cells in order (`Shift + Enter`).
2. Pick an example from the dropdown, or type your own values.
3. Click **Predict** to see the predicted class and the probability of diabetes.

**Requirements:** `ipywidgets` installed, and `models/best_diabetes_model.joblib` created by notebook 03. This notebook must stay in the `app/` folder because the model path is relative.

> This is a teaching project. The output is not medical advice.

## 1. Load the model

In [2]:
# Imports and loading of the saved pipeline (preprocessing + model)
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import joblib
import pandas as pd
import ipywidgets as w
from IPython.display import display

MODEL_PATH = Path("../models/best_diabetes_model.joblib")
model = joblib.load(MODEL_PATH)
print("Modèle chargé :", MODEL_PATH)

Modèle chargé : ..\models\best_diabetes_model.joblib


## 2. Define the form
`EXAMPLES` holds ready-made patients (two come from the dataset). Each input widget matches one feature of the model. Two callbacks drive the form:
- `fill` copies the selected example into the inputs,
- `predict` builds a one-row DataFrame with the exact column names used in training and prints the prediction.

In [4]:
# Example patients, input widgets and the two callbacks (fill, predict)
EXAMPLES = {
    "Saisie libre": dict(Age=50, Gender="M", BMI=28.0, Glucose_Level=120, Blood_Pressure=100,
                         Insulin=150.0, Physical_Activity=60, Family_History="No"),
    "Exemple 1 : patient à risque": dict(Age=58, Gender="F", BMI=36.4, Glucose_Level=151, Blood_Pressure=102,
                                         Insulin=98.5, Physical_Activity=86, Family_History="Yes"),
    "Exemple 2 : patient sain": dict(Age=34, Gender="F", BMI=24.0, Glucose_Level=86, Blood_Pressure=139,
                                     Insulin=181.0, Physical_Activity=46, Family_History="No"),
}

example = w.Dropdown(options=list(EXAMPLES), description="Exemple")
age = w.IntText(description="Âge")
gender = w.Dropdown(options=["F", "M"], description="Genre")
bmi = w.FloatText(description="IMC")
glucose = w.IntText(description="Glucose")
bp = w.IntText(description="Tension")
insulin = w.FloatText(description="Insuline")
activity = w.IntText(description="Activité")
family = w.Dropdown(options=["No", "Yes"], description="Antécédents")
button = w.Button(description="Prédire", button_style="primary")
out = w.Output()

def fill(_=None):
    """Copy the selected example into the input widgets."""
    e = EXAMPLES[example.value]
    age.value, gender.value, bmi.value = e["Age"], e["Gender"], e["BMI"]
    glucose.value, bp.value, insulin.value = e["Glucose_Level"], e["Blood_Pressure"], e["Insulin"]
    activity.value, family.value = e["Physical_Activity"], e["Family_History"]

def predict(_=None):
    """Predict the class and the probability of diabetes for the current inputs."""
    patient = pd.DataFrame([{"Age": age.value, "Gender": gender.value, "BMI": bmi.value,
        "Glucose_Level": glucose.value, "Blood_Pressure": bp.value, "Insulin": insulin.value,
        "Physical_Activity": activity.value, "Family_History": family.value}])
    proba = float(model.predict_proba(patient)[0, 1])
    pred = int(model.predict(patient)[0])
    with out:
        out.clear_output()
        if pred == 1:
            print(f"Prédiction : DIABÉTIQUE (probabilité {proba:.1%})")
        else:
            print(f"Prédiction : NON DIABÉTIQUE (probabilité de diabète {proba:.1%})")

example.observe(fill, names="value")
button.on_click(predict)
fill()

## 3. Display the form

In [5]:
# Show the form: inputs, Predict button and result area
display(w.VBox([example, age, gender, bmi, glucose, bp, insulin, activity, family, button, out]))

## Expected results for the built-in examples
| Example | Expected prediction |
|---|---|
| Example 1 (patient at risk) | Diabetic, probability about 97% |
| Example 2 (healthy patient) | Non-diabetic, probability about 3% |

Small differences are possible, since the model depends on your own train/test split.